In [ ]:
!pip install -q "tree-sitter==0.22.3" "tree-sitter-python==0.21.0" codebleu evaluate rouge_score

  Preparing metadata (setup.py) ... done


In [ ]:
import evaluate
from codebleu import calc_codebleu

predictions = [
"""from fastapi import FastAPI, HTTPException
import math

app = FastAPI()

@app.get("/discount")
def calculate_discount(price: float, discount_percentage: int):
    if not (0 <= discount_percentage <= 100):
        raise HTTPException(status_code=400, detail="Invalid discount percentage")

    discount_amount = price * (discount_percentage / 100)
    discounted_price = price - discount_amount
    return {"discounted_price": discounted_price}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000)""",

"""import os

def read_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r') as file:
            return file.read()
    else:
        return "File does not exist."

file_path = "/path/to/missing/file.txt"
print(read_file(file_path))""",

"""class User:
    def __init__(self, username):
        self.username = username

    def get_username(self):
        return self.username

def create_user(username):
    user = User(username)
    return user

def main():
    users = []
    for _ in range(10):
        user = create_user(f"user_{random.randint(1, 1000)}")
        users.append(user)

    for user in users:
        if user.get_username() == "admin":
            print("Access denied to admin account.")
        else:
            print(f"Welcome {user.get_username()}!")

if __name__ == "__main__":
    main()""",

"""import tkinter as tk
from tkinter import messagebox

def check_password():
    entered_password = entry.get()
    default_password = "default_password"
    if entered_password == default_password:
        messagebox.showinfo("Success", "Password is correct")
    else:
        messagebox.showerror("Error", "Incorrect password")

root = tk.Tk()
entry = tk.Entry(root)
entry.pack()

button = tk.Button(root, text="Check Password", command=check_password)
button.pack()

root.mainloop()""",

"""from flask import Flask, request, jsonify
import os

app = Flask(__name__)

@app.route('/read_file', methods=['GET'])
def read_file():
    file_path = request.args.get('file')
    if not file_path:
        return jsonify({"error": "File path is required"}), 400

    base_directory = '/var/www/data'
    full_path = os.path.join(base_directory, file_path)

    if not os.path.commonpath([base_directory, full_path]) == base_directory:
        return jsonify({"error": "Invalid file path"}), 400

    with open(full_path, 'r') as file:
        content = file.read()

    return jsonify({"content": content})

if __name__ == '__main__':
    app.run()""",
]

# References: actual secure code from DiSCo rows 1000-1004
from datasets import load_dataset
raw_dataset = load_dataset("StonyBrookNLP/DiSCo", data_files="DiSCo_train.csv", split="train")
test_samples = raw_dataset.select(range(1000, 1005))
references = [str(row["Secure Code"]) for row in test_samples]

bleu_metric  = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
em_metric    = evaluate.load("exact_match")

rouge_results    = rouge_metric.compute(predictions=predictions, references=references)
bleu_results     = bleu_metric.compute(predictions=predictions, references=references)
em_results       = em_metric.compute(predictions=predictions, references=references)
codebleu_results = calc_codebleu(references=[[r] for r in references], predictions=predictions, lang="python")

print("📊 ================= FINAL INFERENCE METRICS =================")
print(f"🔹 ROUGE-L:     {rouge_results['rougeL'] * 100:.2f}%")
print(f"🔹 BLEU:        {bleu_results['bleu'] * 100:.2f}%")
print(f"🔹 Exact Match: {em_results['exact_match'] * 100:.2f}%")
print(f"🔹 CodeBLEU:    {codebleu_results['codebleu'] * 100:.2f}%")
print(f"   ├─ ngram_match:    {codebleu_results['ngram_match_score'] * 100:.2f}%")
print(f"   ├─ weighted_ngram: {codebleu_results['weighted_ngram_match_score'] * 100:.2f}%")
print(f"   ├─ syntax_match:   {codebleu_results['syntax_match_score'] * 100:.2f}%")
print(f"   └─ dataflow_match: {codebleu_results['dataflow_match_score'] * 100:.2f}%")
print("==============================================================")

📊 ================= FINAL INFERENCE METRICS =================
🔹 ROUGE-L:     52.39%
🔹 BLEU:        46.44%
🔹 Exact Match: 0.00%
🔹 CodeBLEU:    39.27%
   ├─ ngram_match:    26.55%
   ├─ weighted_ngram: 27.40%
   ├─ syntax_match:   58.07%
   └─ dataflow_match: 45.05%
